# SAiDL Summer Assignment 2026
### Single-notebook execution plan for Core ML + Mechanistic Interpretability

**Execution order:**
1. Setup (once)
2. Core ML — train & benchmark each experiment
3. Mechanistic Interpretability — extract → train SAE → analyse

Each section is independent — you can re-run any cell block without restarting.

## 0  One-time Setup

In [ ]:
# ── 0A  Clone repo ────────────────────────────────────────────────────────────
import os

REPO = '/content/SAiDL-Summer-Assignment-2026'

if not os.path.exists(REPO):
    !git clone https://github.com/VvS-2403/SAiDL-Summer-Assignment-2026.git $REPO
else:
    !cd $REPO && git pull

%cd $REPO
import sys
sys.path.insert(0, REPO)
print('Repo ready at', REPO)

In [ ]:
# ── 0B  Install dependencies ──────────────────────────────────────────────────
!pip install -q \
    transformers datasets wandb hydra-core omegaconf \
    einops tqdm scikit-learn umap-learn matplotlib numpy
print('✅ Packages installed')

In [ ]:
# ── 0C  Create missing config files (idempotent) ──────────────────────────────
import os, textwrap

def write_if_missing(path, content):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        with open(path, 'w') as f:
            f.write(textwrap.dedent(content))
        print(f'  created {path}')

BASE = f'{REPO}/core_ml/configs'

write_if_missing(f'{BASE}/attention/gqa.yaml',
    'name: "gqa"\nnum_heads: 8\nnum_kv_heads: 2\ndropout: 0.1\nis_causal: true\n')

write_if_missing(f'{BASE}/attention/sliding_window.yaml',
    'name: "sliding_window"\nnum_heads: 8\nwindow_size: 256\ndropout: 0.1\nis_causal: true\n')

write_if_missing(f'{BASE}/attention/relu.yaml',
    'name: "relu_attention"\nnum_heads: 8\ndropout: 0.1\nis_causal: true\n')

write_if_missing(f'{BASE}/attention/sparse.yaml',
    'name: "sparse_attention"\nnum_heads: 8\nlocal_window: 64\nstride: 64\ndropout: 0.1\nis_causal: true\n')

write_if_missing(f'{BASE}/positional/rope.yaml',
    'name: "rope"\nmax_len: 1024\nbase: 10000.0\n')

write_if_missing(f'{BASE}/positional/alibi.yaml',
    'name: "alibi"\n')

write_if_missing(f'{BASE}/positional/relative.yaml',
    'name: "relative"\nmax_relative_distance: 128\n')

write_if_missing(f'{BASE}/model/hybrid.yaml', '''
    name: "hybrid_transformer"
    d_model: 512
    n_layers: 6
    n_heads: 8
    d_ff: 2048
    dropout: 0.1
    vocab_size: 50257
    max_seq_len: 1024
    hybrid:
      type: "conv_before_attn"
      conv_kernel_size: 3
      conv_groups: 512
    ''')

# Hybrid __init__.py
hybrid_init = f'{REPO}/core_ml/models/hybrid/__init__.py'
os.makedirs(os.path.dirname(hybrid_init), exist_ok=True)
open(hybrid_init, 'a').close()

print('✅ Configs ready')

In [ ]:
# ── 0D  W&B login ─────────────────────────────────────────────────────────────
import wandb
wandb.login()   # paste API key from https://wandb.ai/authorize

In [ ]:
# ── 0E  Patch sys.path for Hydra (must be done before any train call) ─────────
import sys, os
REPO = '/content/SAiDL-Summer-Assignment-2026'
sys.path.insert(0, REPO)
os.chdir(REPO)

# Patch train.py once so Hydra's CWD change doesn't break imports
PATCH = f'import sys, os\nsys.path.insert(0, "{REPO}")\nos.chdir("{REPO}")\n'
train_py = f'{REPO}/core_ml/train/train.py'
with open(train_py) as f:
    src = f.read()
if 'sys.path.insert' not in src:
    with open(train_py, 'w') as f:
        f.write(PATCH + src)
    print('Patched train.py')
else:
    print('train.py already patched')

## 1  Core ML Experiments

Run each cell to launch one training + benchmark run.
Each takes ~25–35 min on a T4. Run sequentially or pick the ones you need.

In [ ]:
# ── 1.1  Baseline: Vanilla MHA + Sinusoidal ───────────────────────────────────
%cd $REPO
!python core_ml/train/train.py \
    attention=vanilla \
    positional=sinusoidal \
    dataset.batch_size=8 \
    dataset.num_workers=2 \
    experiment_name=baseline_vanilla_sin

In [ ]:
# ── 1.2  Attention variant: Sliding Window + Sinusoidal ───────────────────────
%cd $REPO
!python core_ml/train/train.py \
    attention=sliding_window \
    positional=sinusoidal \
    dataset.batch_size=8 \
    dataset.num_workers=2 \
    experiment_name=attn_sliding_sin

In [ ]:
# ── 1.3  Attention variant: GQA + Sinusoidal ──────────────────────────────────
%cd $REPO
!python core_ml/train/train.py \
    attention=gqa \
    positional=sinusoidal \
    dataset.batch_size=8 \
    dataset.num_workers=2 \
    experiment_name=attn_gqa_sin

In [ ]:
# ── 1.4  Attention variant: ReLU Attention + Sinusoidal ───────────────────────
%cd $REPO
!python core_ml/train/train.py \
    attention=relu \
    positional=sinusoidal \
    dataset.batch_size=8 \
    dataset.num_workers=2 \
    experiment_name=attn_relu_sin

In [ ]:
# ── 1.5  Positional: Vanilla + RoPE ──────────────────────────────────────────
%cd $REPO
!python core_ml/train/train.py \
    attention=vanilla \
    positional=rope \
    dataset.batch_size=8 \
    dataset.num_workers=2 \
    experiment_name=pos_rope

In [ ]:
# ── 1.6  Positional: Vanilla + ALiBi ─────────────────────────────────────────
%cd $REPO
!python core_ml/train/train.py \
    attention=vanilla \
    positional=alibi \
    dataset.batch_size=8 \
    dataset.num_workers=2 \
    experiment_name=pos_alibi

In [ ]:
# ── 1.7  Positional: Vanilla + Relative Bias ─────────────────────────────────
%cd $REPO
!python core_ml/train/train.py \
    attention=vanilla \
    positional=relative \
    dataset.batch_size=8 \
    dataset.num_workers=2 \
    experiment_name=pos_relative

In [ ]:
# ── 1.8  Hybrid: Conv-before-Attn (best attn + best pos from above) ──────────
# Update the attention/positional names below to your best performers.
BEST_ATTN = 'sliding_window'   # ← change to your best
BEST_POS  = 'alibi'            # ← change to your best

%cd $REPO
import subprocess
subprocess.run([
    'python', 'core_ml/train/train.py',
    f'attention={BEST_ATTN}',
    f'positional={BEST_POS}',
    'model=hybrid',
    'model.hybrid.type=conv_before_attn',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=hybrid_conv_before',
], check=True)

In [ ]:
# ── 1.9  Hybrid: Gated Conv FFN ───────────────────────────────────────────────
%cd $REPO
import subprocess
subprocess.run([
    'python', 'core_ml/train/train.py',
    f'attention={BEST_ATTN}',
    f'positional={BEST_POS}',
    'model=hybrid',
    'model.hybrid.type=gated_conv_ffn',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=hybrid_gated_conv',
], check=True)

In [ ]:
# ── 1.10  Benchmark all checkpoints (perplexity + latency at 512/1024) ────────
# Run once after all training is done. Reads checkpoints from outputs/
# and logs a summary table to W&B.

import glob, json

BENCH_PATCH = f'import sys, os\nsys.path.insert(0, "{REPO}")\nos.chdir("{REPO}")\n'
bench_py = f'{REPO}/core_ml/experiments/benchmark.py'
with open(bench_py) as f:
    src = f.read()
if 'sys.path.insert' not in src:
    with open(bench_py, 'w') as f:
        f.write(BENCH_PATCH + src)

# Run benchmark for vanilla baseline (repeat for others as needed)
!python core_ml/experiments/benchmark.py \
    attention=vanilla \
    positional=sinusoidal \
    dataset.batch_size=4 \
    dataset.num_workers=2

## 2  Mechanistic Interpretability

In [ ]:
# ── 2.1  Extract Layer-3 activations from distilgpt2 ─────────────────────────
# Patches extract script path, then runs it.
import os
REPO = '/content/SAiDL-Summer-Assignment-2026'
os.chdir(REPO)

EXTRACT_PATCH = f'import sys, os\nsys.path.insert(0, "{REPO}")\nos.chdir("{REPO}")\n'
ext_py = f'{REPO}/mechanistic_interpretability/pipeline/extract_activations.py'
with open(ext_py) as f:
    src = f.read()
if 'sys.path.insert' not in src:
    with open(ext_py, 'w') as f:
        f.write(EXTRACT_PATCH + src)

!python mechanistic_interpretability/pipeline/extract_activations.py \
    pipeline.batch_size=32 \
    data.max_samples=200000

In [ ]:
# ── 2.2  Train SAE ────────────────────────────────────────────────────────────
import os
REPO = '/content/SAiDL-Summer-Assignment-2026'
os.chdir(REPO)

SAE_PATCH = f'import sys, os\nsys.path.insert(0, "{REPO}")\nos.chdir("{REPO}")\n'
sae_py = f'{REPO}/mechanistic_interpretability/pipeline/train_sae.py'
with open(sae_py) as f:
    src = f.read()
if 'sys.path.insert' not in src:
    with open(sae_py, 'w') as f:
        f.write(SAE_PATCH + src)

!python mechanistic_interpretability/pipeline/train_sae.py \
    pipeline.batch_size=64 \
    sae.k=32 \
    sae.expansion_factor=4 \
    training.num_epochs=1

In [ ]:
# ── 2.3  Evaluate SAE at multiple quantization bit-widths ─────────────────────
import os
REPO = '/content/SAiDL-Summer-Assignment-2026'
os.chdir(REPO)

EVAL_PATCH = f'import sys, os\nsys.path.insert(0, "{REPO}")\nos.chdir("{REPO}")\n'
eval_py = f'{REPO}/mechanistic_interpretability/analysis/evaluate.py'
with open(eval_py) as f:
    src = f.read()
if 'sys.path.insert' not in src:
    with open(eval_py, 'w') as f:
        f.write(EVAL_PATCH + src)

for bits in [8, 4]:
    print(f'\n=== Evaluating {bits}-bit ===')
    !python mechanistic_interpretability/analysis/evaluate.py \
        quantization.bits={bits} \
        quantization.method=uniform_affine \
        pipeline.batch_size=32

In [ ]:
# ── 2.4  Spectral analysis (SVD of SAE decoder) ───────────────────────────────
import os
REPO = '/content/SAiDL-Summer-Assignment-2026'
os.chdir(REPO)

SPEC_PATCH = f'import sys, os\nsys.path.insert(0, "{REPO}")\nos.chdir("{REPO}")\n'
spec_py = f'{REPO}/mechanistic_interpretability/analysis/spectral.py'
with open(spec_py) as f:
    src = f.read()
if 'sys.path.insert' not in src:
    with open(spec_py, 'w') as f:
        f.write(SPEC_PATCH + src)

for bits in [32, 8, 4]:
    print(f'\n=== Spectral {bits}-bit ===')
    !python mechanistic_interpretability/analysis/spectral.py \
        quantization.bits={bits}

In [ ]:
# ── 2.5  UMAP visualization ───────────────────────────────────────────────────
import os
REPO = '/content/SAiDL-Summer-Assignment-2026'
os.chdir(REPO)

VIZ_PATCH = f'import sys, os\nsys.path.insert(0, "{REPO}")\nos.chdir("{REPO}")\n'
viz_py = f'{REPO}/mechanistic_interpretability/analysis/visualization.py'
with open(viz_py) as f:
    src = f.read()
if 'sys.path.insert' not in src:
    with open(viz_py, 'w') as f:
        f.write(VIZ_PATCH + src)

!python mechanistic_interpretability/analysis/visualization.py \
    quantization.bits=4

In [ ]:
# ── 2.6  Neuron analysis (top activating tokens per feature) ──────────────────
import os
REPO = '/content/SAiDL-Summer-Assignment-2026'
os.chdir(REPO)

NEURO_PATCH = f'import sys, os\nsys.path.insert(0, "{REPO}")\nos.chdir("{REPO}")\n'
neuro_py = f'{REPO}/mechanistic_interpretability/analysis/neuron_analysis.py'
with open(neuro_py) as f:
    src = f.read()
if 'sys.path.insert' not in src:
    with open(neuro_py, 'w') as f:
        f.write(NEURO_PATCH + src)

!python mechanistic_interpretability/analysis/neuron_analysis.py \
    quantization.bits=8

In [ ]:
# ── 2.7  Semantic clustering ──────────────────────────────────────────────────
import os
REPO = '/content/SAiDL-Summer-Assignment-2026'
os.chdir(REPO)

CLUS_PATCH = f'import sys, os\nsys.path.insert(0, "{REPO}")\nos.chdir("{REPO}")\n'
clus_py = f'{REPO}/mechanistic_interpretability/analysis/clustering.py'
with open(clus_py) as f:
    src = f.read()
if 'sys.path.insert' not in src:
    with open(clus_py, 'w') as f:
        f.write(CLUS_PATCH + src)

!python mechanistic_interpretability/analysis/clustering.py \
    quantization.bits=8

## 3  Download all outputs for the report

In [ ]:
# Zip everything in outputs/ and benchmark_results/ for download
import os
REPO = '/content/SAiDL-Summer-Assignment-2026'

!cd $REPO && zip -r /content/saidl_outputs.zip outputs benchmark_results \
    mechanistic_interpretability/outputs 2>/dev/null || true

from google.colab import files
files.download('/content/saidl_outputs.zip')
print('Download started.')